# Phase 1 — TTT-Linear 30M on Colab T4 (from scratch)

Same data, tokenizer, width/depth, **AdamW L2 0.1**, cross-entropy. Mixer is `ttt_linear`.

Compare vs **attention val PPL 5.23** (old snapshot), not epoch-19 `last.pt`. Do **not** pass `baseline-30m/last.pt`.

1. Runtime → **T4 GPU**.
2. Use a **new** Colab runtime (not the attention continue-fit).
3. Zip current `prototype/` to Drive (same as the attention notebook).

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then reconnect."
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/ttt-prototype")
DRIVE.mkdir(parents=True, exist_ok=True)
print("Checkpoints will be saved under", DRIVE / "checkpoints")

In [ ]:
REPO = "https://github.com/YOUR_USER/YOUR_REPO.git"
USE_ZIP = True
ZIP_ON_DRIVE = "/content/drive/MyDrive/ttt-prototype/prototype.zip"

if USE_ZIP:
    !unzip -q -o {ZIP_ON_DRIVE} -d /content
    %cd /content/prototype
else:
    !git clone --depth 1 {REPO} /content/repo
    %cd /content/repo/new_llm_architecture-main/prototype

In [ ]:
%pip install -q tokenizers numpy
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

In [ ]:
from pathlib import Path
import shutil

drive_data = Path("/content/drive/MyDrive/ttt-prototype/data")
data_dir = Path("data/tinystories-v2")
data_dir.mkdir(parents=True, exist_ok=True)
Path("data/tokenizer").mkdir(parents=True, exist_ok=True)

for name in ("train.npy", "validation.npy"):
    src = drive_data / "tinystories-v2" / name
    dst = data_dir / name
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
tok_src = drive_data / "tokenizer" / "tokenizer.json"
tok_dst = Path("data/tokenizer/tokenizer.json")
if tok_src.exists() and not tok_dst.exists():
    shutil.copy(tok_src, tok_dst)

if not (data_dir / "train.npy").exists():
    !python scripts/prepare_tinystories.py --train-mib 100 --validation-mib 10
    !python scripts/train_tokenizer.py
    !python scripts/encode_corpus.py --input data/tinystories-v2/train.jsonl --output data/tinystories-v2/train.npy
    !python scripts/encode_corpus.py --input data/tinystories-v2/validation.jsonl --output data/tinystories-v2/validation.npy
print("data ok", (data_dir / "train.npy").stat().st_size)

In [ ]:
from pathlib import Path

FROM_SCRATCH = True  # Phase 1. Set False only to continue this TTT run after a disconnect.
out = Path("/content/drive/MyDrive/ttt-prototype/checkpoints/ttt-linear-30m")
out.mkdir(parents=True, exist_ok=True)
resume = out / "last.pt"
WEIGHT_DECAY = 0.1
print("FROM_SCRATCH", FROM_SCRATCH, "existing last.pt", resume.exists())

if FROM_SCRATCH:
    !python -u scripts/train.py --mixer ttt_linear --epochs 10 --optimizer adamw \
      --weight-decay {WEIGHT_DECAY} --dropout 0.1 --from-scratch --device cuda --amp auto \
      --seq-len 256 --batch-size 2 --grad-accum 2 \
      --out {out}
else:
    resume_flag = f"--resume {resume}" if resume.exists() else ""
    !python -u scripts/train.py --mixer ttt_linear --epochs 10 --optimizer adamw \
      --weight-decay {WEIGHT_DECAY} --dropout 0.1 --device cuda --amp auto \
      --seq-len 256 --batch-size 2 --grad-accum 2 \
      --out {out} {resume_flag}